# Does the GFTM extent / parity-filter result reproduce on TGLF?

Fusion_PhD-38e9.4. Repeats 38e9.1 (extent cutoff on the dominant-mode selection) and 38e9.2 (parity gate, the
`FILTER=0.5` equivalent) on **TGLF**, SPR-045 and M1, side by side with GFTM from the same notebook.

Data: TGLF width-oracle cubes (`wo_*`; registry `tglf_width_oracle`: FIND_WIDTH F, NBASIS_MAX 4, NXGRID 16,
THETA_TRAPPED 0.7, **FILTER 0.0**, NMODES 4, WRITE_WAVEFUNCTION_FLAG 1), 24 widths 0.2-4.0. GFTM `wo_*` (22/28, FILTER 0.0)
and `wo_f05_*` (FILTER 0.5) for the comparison. GS2 reference `pyro_cube_avg`. Same population rule as 38e9.1:
GS2 growth rate finite and > 0.

**TGLF's FILTER (source, `gacode/tglf/src`, upstream commit 70cbc41)**: `FILTER > 0` is a *frequency* filter
(`tglf_eigensolver.f90:2918`, an unstable mode with |omega_i| > FILTER*|ky|*max_freq is flipped stable); `FILTER = 0` does
nothing; **`FILTER < 0` is a parity gate** (`tglf_LS.f90:272-419`): a mode whose phi Hermite weight is >85% in the odd-parity
basis functions is removed (always if |FILTER| <= 0.1, else only if |omega| > |FILTER|*|ky|*max_freq).
GFTM's gate removes a mode when phi_norm_even < FILTER (gftm_LS.f90:582). So TGLF's gate is selected by the **sign** of FILTER.

**The oracle (mode closest to GS2) consults the answer: it is a bound, never an accuracy.**

## Imports and settings

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import pyrokinetics
from pyrokinetics.pyroscan import PyroScanGKOutput
from pyrokinetics.dataset_wrapper import DatasetWrapper
from pyrokinetics.diagnostics.extent import Extent

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir()
)
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)
print("pyrokinetics", pyrokinetics.__version__)

analysis_name = "kbm_extent_filter_tglf"
data_root = Path(os.environ["GK_DATA_ROOT"]).expanduser()
run_template = "Runs"
project = "LATIN_HYPERCUBE"
cases = {"SPR-045": "SPR-045", "M1": "M1"}  # database label -> case directory
scan_information = {"TGLF": "kbm_8d/wo_{width}", "GFTM": "kbm_8d/wo_{width}", "GFTM05": "kbm_8d/wo_f05_{width}", "GS2": "kbm_8d"}
output_dir = {"TGLF": "pyro_cube", "GFTM": "pyro_cube", "GS2": "pyro_cube_avg"}  # GS2: tail-averaged reference
output_file = "cube.nc"
widths = ["w0p2000", "w0p2297", "w0p2639", "w0p3031", "w0p3482", "w0p4000", "w0p4595",
          "w0p5279", "w0p6063", "w0p6965", "w0p8000", "w0p9000", "w1p0000", "w1p1000",
          "w1p2000", "w1p3000", "w1p4000", "w1p5000", "w1p6000", "w1p9218", "w2p3083",
          "w2p7726", "w3p3302", "w4p0000"]
width_value = {w: float(w[1:].replace("p", ".")) for w in widths}
resolution = {"TGLF": {"nbasis_used": 4, "nxgrid_used": 16}, "GFTM": {"nbasis_used": 22, "nxgrid_used": 28}}

fraction = 0.95                             # Extent's rule: theta range holding 95% of |Re field|^2
fields = ["phi", "apar"]                    # phi selects; apar reported alongside
cutoffs = np.round(np.arange(0, 12.01, 0.25), 2)       # absolute extent cutoff [rad]
rel_cutoffs = np.round(np.arange(0, 1.001, 0.05), 2)   # fraction of the case's largest unstable-mode extent
edge_amplitude = 0.05    # check 1: |phi| at the theta-grid edge above this fraction of its peak
delta_lim = 0.2          # fig 2 colour range for the medlog change (clipped; misses-dominate cells are grey)
example_width = "w1p0000"
n_examples = 3

## Load data

One `PyroScanGKOutput` per (code, database, width) from the cube netCDF; `Extent` adds extent/bounds in place.

In [ ]:
gs2, tglf, gftm, gftm05 = {}, {}, {}, {}
def load(code_, w, key="TGLF"):
    out = PyroScanGKOutput.from_netcdf(data_root / code_ / run_template / project / case / scan_information[key].format(width=w) / "pyro_cube" / output_file)
    if code_ == "TGLF":
        Extent(out, fraction=fraction)   # extents only for the TGLF analysis; GFTM cubes are references
    return out
for db, case in cases.items():
    gs2[db] = PyroScanGKOutput.from_netcdf(
        data_root / "GS2" / run_template / project / case / scan_information["GS2"] / output_dir["GS2"] / output_file)
    for w in widths:
        tglf[db, w] = load("TGLF", w)
        gftm[db, w] = load("GFTM", w, "GFTM")
        gftm05[db, w] = load("GFTM", w, "GFTM05")
    print(db, "samples", gs2[db].data.sizes["sample"], "| cubes per arm", len(widths))

## Calculate and select

Conventions, checked rather than assumed: both codes' growth rates are in the same pyrokinetics
units; each cube's audited resolution (`*_used`, from `out.gftm.localdump`) is the requested one.

Per case, the code returns 4 modes. A **selector** takes the argmax growth rate over `mode` among
*unstable* modes (gamma > 0) that pass its gate. A case where no unstable mode passes is a **MISS**:
it stays in the population with gamma_sel = 0, so its |log10| error is +inf and its absolute error is
gamma_GS2. Misses therefore move the median, and medlog becomes inf once they are the majority.

Population: cases whose GS2 growth rate is finite and positive (the same cases for every selector
and width, so every comparison is paired).

- `dominant`: no gate (c = 0; the current argmax over unstable modes).
- `oracle`: the unstable mode with the smallest |log10(gamma/gamma_GS2)| -- a bound.
- `extent >= c`: absolute cutoff on the 95% extent of phi (apar alongside).
- `extent >= r * max extent`: relative cutoff on the case's own longest unstable mode.

In [ ]:
def mag(v):
    return np.asarray(getattr(v.data, "magnitude", v.data))


def select(gam, keep):
    # argmax growth over 'mode' among unstable modes passing `keep`; 0 where none passes (a MISS)
    g = np.where(keep & (gam > 0), gam, -np.inf)
    idx = g.argmax(axis=1)
    return np.where(np.isfinite(g.max(axis=1)), gam[np.arange(len(gam)), idx], 0.0), idx


def score(g_sel, g_ref):
    with np.errstate(divide="ignore"):
        log_err = np.abs(np.log10(g_sel / g_ref))
    return np.median(np.abs(g_sel - g_ref)), np.median(log_err), int((g_sel == 0).sum())




def analyse(code_, cubes):
    rows, sweep, cases_data = [], [], {}
    for (db, w), out in cubes.items():
        d, ref = out.data, gs2[db].data
        assert str(d.growth_rate.data.units) == str(ref.growth_rate.data.units), "normalisations differ"
        for var, value in resolution[code_].items():
            assert (mag(d[var]) == value).all(), f"{db}/{w}: {var} is not {value}"
        assert np.allclose(mag(d.width_used), width_value[w])

        ref_by_name = dict(zip(ref.sample_name.values, mag(ref.growth_rate)))
        g_ref_all = np.array([ref_by_name.get(n, np.nan) for n in d.sample_name.values])
        ok = np.isfinite(g_ref_all) & (g_ref_all > 0)
        g_ref = g_ref_all[ok]
        gam = mag(d.growth_rate.transpose("sample", "mode"))[ok]
        ext = {f: mag(d.extent.sel(field=f).transpose("sample", "mode"))[ok] for f in fields}
        theta = d.theta.values
        phi = mag(d.eigenfunctions.sel(field="phi").transpose("sample", "theta", "mode"))[ok]

        unstable = gam > 0
        g_dom, i_dom = select(gam, True)
        with np.errstate(divide="ignore", invalid="ignore"):
            dist = np.where(unstable, np.abs(np.log10(gam / g_ref[:, None])), np.inf)
        i_orc = dist.argmin(axis=1)
        g_orc = np.where(unstable.any(axis=1), gam[np.arange(len(gam)), i_orc], 0.0)

        # check 1: 95% bounds within one grid spacing of the edge, or |phi| at the edge above edge_amplitude of peak
        lo, hi = (mag(d.bounds.sel(field="phi", bound=b).transpose("sample", "mode"))[ok] for b in ("lo", "hi"))
        dtheta = theta[1] - theta[0]
        at_edge = (lo <= theta[0] + dtheta) | (hi >= theta[-1] - dtheta)
        edge_amp = np.maximum(np.abs(phi[:, 0]), np.abs(phi[:, -1])) / np.abs(phi).max(axis=1)

        # check 2: rotate each eigenfunction so its peak is real, recompute Extent
        e = mag(d.eigenfunctions)
        axis = d.eigenfunctions.get_axis_num("theta")
        peak = np.take_along_axis(e, np.abs(e).argmax(axis=axis, keepdims=True), axis=axis)
        rot = DatasetWrapper(data_vars={"eigenfunctions": d.eigenfunctions.copy(data=e * np.exp(-1j * np.angle(peak)))})
        Extent(rot, fraction=fraction)
        with np.errstate(divide="ignore", invalid="ignore"):
            phase_change = {f: np.abs(mag(rot.data.extent.sel(field=f).transpose("sample", "mode"))[ok] - ext[f]) / ext[f]
                            for f in fields}
        del rot

        differ = i_dom != i_orc
        with np.errstate(divide="ignore"):
            err_dom, err_orc = np.abs(np.log10(g_dom / g_ref)), np.abs(np.log10(g_orc / g_ref))
        r = np.arange(len(gam))
        row = dict(db=db, width=width_value[w], n=int(ok.sum()), n_gs2_stable_or_nan=int((~ok).sum()),
                   differ=int(differ.sum()),
                   medlog_dom=score(g_dom, g_ref)[1], medlog_orc=score(g_orc, g_ref)[1],
                   medAE_dom=score(g_dom, g_ref)[0], medAE_orc=score(g_orc, g_ref)[0],
                   medlog_dom_agree=np.median(err_dom[~differ]), medlog_dom_differ=np.median(err_dom[differ]),
                   medlog_orc_differ=np.median(err_orc[differ]),
                   share_err_from_differ=err_dom[differ & np.isfinite(err_dom)].sum() / err_dom[np.isfinite(err_dom)].sum(),
                   unstable_modes=int(unstable.sum()), unstable_nan_extent=int((unstable & ~np.isfinite(ext["phi"])).sum()),
                   frac_bounds_at_edge=at_edge[unstable].mean(), frac_edge_amp=(edge_amp[unstable] > edge_amplitude).mean(),
                   **{f"phase_{f}_median_rel_change": np.nanmedian(phase_change[f][unstable]) for f in fields},
                   **{f"phase_{f}_frac_over_10pct": (phase_change[f][unstable] > 0.1).mean() for f in fields})
        for f in fields:
            a, b = ext[f][r, i_dom][differ], ext[f][r, i_orc][differ]
            row[f"{f}_ext_dom_shorter"] = (a < b).mean()
            row[f"{f}_ext_ratio_median"] = np.median(a / b)
        rows.append(row)

        longest = np.nanmax(np.where(unstable, ext["phi"], np.nan), axis=1, initial=0)
        for f in fields:
            for kind, grid in (("abs", cutoffs), ("rel", rel_cutoffs)):
                for c in grid:
                    threshold = c if kind == "abs" else c * longest[:, None]
                    keep = (ext[f] >= threshold) | (c == 0)   # c = 0 keeps NaN-extent modes: exactly `dominant`
                    medAE, medlog, miss = score(select(gam, keep)[0], g_ref)
                    sweep.append(dict(db=db, width=width_value[w], field=f, kind=kind, cutoff=c,
                                      medlog=medlog, medAE=medAE, miss=miss, n=len(g_ref)))
        cases_data[db, w] = dict(names=d.sample_name.values[ok], gam=gam, g_ref=g_ref, i_dom=i_dom, i_orc=i_orc,
                                 ext=ext, err_dom=err_dom, differ=differ)

    return pd.DataFrame(rows), pd.DataFrame(sweep), cases_data


In [ ]:
summary, sweep, cases_data = analyse("TGLF", tglf)


### (a) The premise, measured

`differ`: cases where the oracle mode is not the dominant one. `share_err_from_differ`: fraction of
the dominant arm's total |log10| error carried by those cases.

In [ ]:
pd.set_option("display.width", 250, "display.max_columns", 40)
print(summary[["db", "width", "n", "n_gs2_stable_or_nan", "differ", "medlog_dom", "medlog_orc",
               "medlog_dom_agree", "medlog_dom_differ", "medlog_orc_differ", "share_err_from_differ",
               "medAE_dom", "medAE_orc"]].round(4).to_string(index=False))
print(summary.groupby("db")[["differ", "share_err_from_differ"]].agg(["min", "median", "max"]).round(3))

### Checks on the extent numbers (grid edge, phase, NaN extents)

In [ ]:
print(summary[["db", "width", "unstable_modes", "unstable_nan_extent", "frac_bounds_at_edge", "frac_edge_amp",
               ] + [f"phase_{f}_{k}" for f in fields for k in ("median_rel_change", "frac_over_10pct")]].round(4).to_string(index=False))

### (b) Extent of the spurious dominant mode vs the oracle mode, where they differ

In [ ]:
print(summary[["db", "width", "differ"] + [f"{f}_{k}" for f in fields for k in ("ext_dom_shorter", "ext_ratio_median")]]
      .round(3).to_string(index=False))
for db in cases:
    for f in fields:
        a = np.concatenate([v["ext"][f][np.arange(len(v["gam"])), v["i_dom"]][v["differ"]] for (d_, w), v in cases_data.items() if d_ == db])
        b = np.concatenate([v["ext"][f][np.arange(len(v["gam"])), v["i_orc"]][v["differ"]] for (d_, w), v in cases_data.items() if d_ == db])
        print(f"{db} {f}: pooled over widths n={len(a)}, dominant shorter in {np.mean(a < b):.3f}, "
              f"median ratio dom/orc {np.nanmedian(a / b):.3f}")

### (c) The cutoff sweep

For each (database, width): the cutoff minimising medlog, its miss count and change vs c = 0,
next to the oracle bound. The parity baseline (d) is deferred to pyro#594.

In [ ]:
best = (sweep[np.isfinite(sweep.medlog)].sort_values("medlog").groupby(["db", "field", "kind", "width"]).head(1)
        .set_index(["db", "field", "kind", "width"]).sort_index())
base = sweep[sweep.cutoff == 0].set_index(["db", "field", "kind", "width"])
best = best.assign(medlog_c0=base.medlog, medAE_c0=base.medAE)
best = best.assign(d_medlog=best.medlog - best.medlog_c0)
best = best.join(summary.set_index(["db", "width"])[["medlog_orc"]])
for f in fields:
  print(f"--- {f}, absolute cutoff ---")
  print(best.loc[(slice(None), f, "abs"), ["cutoff", "medlog", "medlog_c0", "d_medlog", "medAE", "medAE_c0",
                                             "miss", "medlog_orc"]].round(4).to_string())
print("--- phi and apar, relative cutoff ---")
print(best.loc[(slice(None), slice(None), "rel"), ["cutoff", "medlog", "medlog_c0", "d_medlog", "miss"]].round(4).to_string())
for f in fields:
    for kind in ("abs", "rel"):
        b = best.loc[(slice(None), f, kind)]
        wins = b[b.d_medlog < 0]
        print(f"{f}/{kind}: cells where some cutoff lowers medlog: "
              + ", ".join(f"{db} {len(wins.loc[db]) if db in wins.index.get_level_values(0) else 0}/{len(widths)}" for db in cases)
              + f" | largest improvement: " + ", ".join(f"{db} {b.loc[db].d_medlog.min():+.4f}" for db in cases))

In [ ]:
plt.rcParams.update({"axes.titlesize": 16, "axes.labelsize": 16, "xtick.labelsize": 14, "ytick.labelsize": 14, "legend.fontsize": 12})
tick = [0.5, 1, 2, 5, 10]
field_label = {"phi": r"$\phi$", "apar": r"$A_\parallel$"}

## Filter arms on TGLF

Four TGLF arms on the same `wo_*` decks (GyroRun `from_directory` + `code_flags`, tree `kbm_tglf_filter`; only FILTER differs): `fp0p0` FILTER=0.0
(no filter), `fm0p1` FILTER=-0.1 (native parity gate), `fp2p0` FILTER=2.0 (native frequency filter). `fp0p0` also carries pyro's per-run
`phi_even_fraction` and `apar_even_fraction` (`load_tearing_parameter=True`, each run's own geometry). Post-hoc gates on `fp0p0`:
**phi gate**: drop modes with phi_even_fraction < 0.5 (GFTM's FILTER=0.5 criterion); **A_par gate**: drop modes with apar_even_fraction >= 0.5
(tearing parity). Dominant mode = argmax growth among surviving unstable modes; no survivor = miss (growth 0, frequency 0, kept in the population).
Metric: median absolute error against GS2 in the shared pyro units, plotted as log10. GFTM `wo_*` / `wo_f05_*` for reference.
The oracle (mode closest to GS2) is a bound.

In [ ]:
arm_tag = {"f00": "fp0p0", "nat": "fm0p1", "f2": "fp2p0"}
arm = {}
for db, case in cases.items():
    for w in widths:
        for k, t in arm_tag.items():
            arm[k, db, w] = PyroScanGKOutput.from_netcdf(data_root / "TGLF" / run_template / "kbm_tglf_filter" / case / f"wo_{t}_{w}" / "pyroscan_tearing.nc")
            Extent(arm[k, db, w], fraction=fraction)
    Extent(gs2[db], fraction=fraction)
print(len(arm), "arm cubes; variables:", list(arm["f00", "SPR-045", widths[0]].data.data_vars))
print({v: arm["f00", "SPR-045", widths[0]].data[v].dims for v in ("growth_rate", "mode_frequency", "phi_even_fraction", "apar_even_fraction")})

In [ ]:
def grab(out, name, names):
    v = out.data[name]
    other = [d for d in v.dims if d not in ("sample", "mode")]
    a = mag(v.transpose("sample", *other, "mode")).astype(float)
    if other:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            a = np.nanmax(a, axis=tuple(range(1, a.ndim - 1)))
    idx = pd.Index(out.data.sample_name.values).get_indexer(names)
    assert (idx >= 0).all()
    return a[idx]


def pick(gam, freq, keep=True):
    g, i = select(gam, keep)
    f = np.where(g > 0, freq[np.arange(len(gam)), i], 0.0)
    return g, f


def medae(x, ref):
    return float(np.median(np.abs(x - ref)))


rows, removed = [], []
for db in cases:
    ref = gs2[db].data
    gby, fby = dict(zip(ref.sample_name.values, mag(ref.growth_rate))), dict(zip(ref.sample_name.values, mag(ref.mode_frequency)))
    for w in widths:
        d0 = arm["f00", db, w]
        all_names = d0.data.sample_name.values
        g_all = np.array([gby.get(n, np.nan) for n in all_names])
        ok = np.isfinite(g_all) & (g_all > 0)
        names, g_ref = all_names[ok], g_all[ok]
        f_ref = np.array([fby[n] for n in names])
        assert np.isfinite(f_ref).all()
        data = {k: {"g": grab(arm[k, db, w], "growth_rate", names), "f": grab(arm[k, db, w], "mode_frequency", names)} for k in arm_tag}
        phi_e, apar_e = grab(d0, "phi_even_fraction", names), grab(d0, "apar_even_fraction", names)
        g0, f0 = data["f00"]["g"], data["f00"]["f"]
        unstable = g0 > 0
        sel = {"f00": pick(g0, f0), "phi": pick(g0, f0, phi_e >= 0.5), "apar": pick(g0, f0, apar_e < 0.5),
               "nat": pick(data["nat"]["g"], data["nat"]["f"]), "f2": pick(data["f2"]["g"], data["f2"]["f"])}
        for k, key in (("gftm0", gftm), ("gftm05", gftm05)):
            o = key[db, w]
            sel[k] = pick(grab(o, "growth_rate", names), grab(o, "mode_frequency", names))
        with np.errstate(divide="ignore", invalid="ignore"):
            io = np.where(unstable, np.abs(np.log10(g0 / g_ref[:, None])), np.inf).argmin(axis=1)
            fo = np.where(unstable, np.abs(f0 - f_ref[:, None]), np.inf).argmin(axis=1)
        r = np.arange(len(g0))
        row = dict(db=db, width=width_value[w], n=len(names), medAE_g_orc=medae(np.where(unstable.any(1), g0[r, io], 0.0), g_ref),
                   medAE_f_orc=medae(np.where(unstable.any(1), f0[r, fo], 0.0), f_ref))
        for k, (g, f) in sel.items():
            row.update({f"medAE_g_{k}": medae(g, g_ref), f"medAE_f_{k}": medae(f, f_ref), f"miss_{k}": int((g == 0).sum())})
        rows.append(row)
        # which unstable FILTER=0 modes does each filter remove? native arms: not present (same growth AND frequency) afterwards
        def present(k):
            gx, fx = data[k]["g"], data[k]["f"]
            return np.array([[(np.isclose(gx[s], g0[s, m], rtol=1e-6, atol=1e-9) & np.isclose(fx[s], f0[s, m], rtol=1e-6, atol=1e-9)).any() for m in range(g0.shape[1])] for s in r])
        gone = {"phi": unstable & ~(phi_e >= 0.5), "apar": unstable & ~(apar_e < 0.5), "nat": unstable & ~present("nat"), "f2": unstable & ~present("f2")}
        rec = dict(db=db, width=width_value[w], n_unstable=int(unstable.sum()), **{f"gone_{k}": int(v.sum()) for k, v in gone.items()})
        rec.update(both=int((gone["phi"] & gone["nat"]).sum()), only_phi=int((gone["phi"] & ~gone["nat"]).sum()),
                   only_nat=int((~gone["phi"] & gone["nat"]).sum()), neither=int((unstable & ~gone["phi"] & ~gone["nat"]).sum()))
        removed.append(rec)
arms_df, removed_df = pd.DataFrame(rows), pd.DataFrame(removed)
pd.set_option("display.width", 250, "display.max_columns", 60)
print(arms_df.round(4).to_string(index=False))
print(removed_df.to_string(index=False))

In [ ]:
from matplotlib.ticker import StrMethodFormatter
tick = [0.2, 0.5, 1, 2, 4]
def style_x(ax):
    ax.set_xscale("log"); ax.set_xticks(tick); ax.xaxis.set_major_formatter(StrMethodFormatter("{x:g}")); ax.minorticks_off()
    ax.set_xlim(min(widths_f), max(widths_f))
widths_f = [width_value[w] for w in widths]
lines = {"f00": ("FILTER 0", dict(color="C0")), "nat": (r"parity gate (FILTER $-$0.1)", dict(color="C3")), "f2": ("FILTER 2", dict(color="C1")),
         "orc": ("oracle", dict(color="0.5", ls=":"))}
ylab = {"f": r"LogMed $|\omega_{\rm TGLF}-\omega_{\rm GS2}|$", "g": r"LogMed $|\gamma_{\rm TGLF}-\gamma_{\rm GS2}|$"}
fig5, axes5 = plt.subplots(2, len(cases), figsize=(14, 10), sharex=True)
for col, db in enumerate(cases):
    s = arms_df[arms_df.db == db]
    for row, m in enumerate(("f", "g")):
        ax = axes5[row, col]
        for k, (lab, st) in lines.items():
            ax.plot(s.width, np.log10(s[f"medAE_{m}_{k}"]), marker="o" if k != "orc" else "", ms=3, label=lab, **st)
        style_x(ax)
        ax.set_ylabel(ylab[m])
    axes5[0, col].set_title(db)
    axes5[1, col].set_xlabel("WIDTH")
axes5[1, 0].legend()
plt.show()

In [ ]:
tab = []
for db in cases:
    s, r_ = arms_df[arms_df.db == db], removed_df[removed_df.db == db]
    for k in ("phi", "apar", "nat", "f2"):
        for m in ("g", "f"):
            a, b, o = s[f"medAE_{m}_{'gftm0' if k == 'gftm05' else 'f00'}"], s[f"medAE_{m}_{k}"], s[f"medAE_{m}_orc"] if k != "gftm05" else None
            tab.append(dict(db=db, arm=k, metric=m, widths_better=int((b < a).sum()), widths_worse=int((b > a).sum()), widths=len(s),
                            median_dlog10=float(np.median(np.log10(b / a))), misses=int(s[f"miss_{k}"].sum()),
                            modes_removed=int(r_[f"gone_{k}"].sum()) if f"gone_{k}" in r_ else np.nan, unstable_modes=int(r_.n_unstable.sum())))
summ = pd.DataFrame(tab)
print(summ.round(3).to_string(index=False))
ov = removed_df.groupby("db")[["both", "only_phi", "only_nat", "neither"]].sum()
print(ov)

### Extent of the selected mode relative to GS2

Median over cases of log10(extent of the selected TGLF mode / extent of the GS2 mode), 95% rule of pyro `Extent`, per arm; misses are left out.

In [ ]:
def ext_of(out, field, names):
    e = out.data.extent.sel(field=field)
    a = mag(e.transpose("sample", "mode")) if "mode" in e.dims else mag(e)[:, None]
    return a[pd.Index(out.data.sample_name.values).get_indexer(names)]


ext_rows = []
for db in cases:
    ref = gs2[db].data
    gby = dict(zip(ref.sample_name.values, mag(ref.growth_rate)))
    for w in widths:
        d0 = arm["f00", db, w]
        all_names = d0.data.sample_name.values
        g_all = np.array([gby.get(n, np.nan) for n in all_names])
        ok = np.isfinite(g_all) & (g_all > 0)
        names = all_names[ok]
        phi_e = grab(d0, "phi_even_fraction", names)
        for f in fields:
            e_ref = ext_of(gs2[db], f, names)[:, 0]
            e0 = ext_of(d0, f, names)
            gam = {k: grab(arm[k, db, w], "growth_rate", names) for k in arm_tag}
            keep = {"f00": True, "phi": phi_e >= 0.5, "nat": True, "f2": True}
            for k, src in (("f00", "f00"), ("phi", "f00"), ("nat", "nat"), ("f2", "f2")):
                g, i = select(gam[src], keep[k])
                e = (e0 if src == "f00" else ext_of(arm[src, db, w], f, names))[np.arange(len(names)), i]
                with np.errstate(divide="ignore", invalid="ignore"):
                    ext_rows.append(dict(db=db, width=width_value[w], field=f, arm=k, log_ratio=float(np.nanmedian(np.where(g > 0, np.log10(e / e_ref), np.nan)))))
ext_df = pd.DataFrame(ext_rows)
ext_label = {"f00": "FILTER 0", "nat": r"parity gate (FILTER $-$0.1)", "f2": "FILTER 2"}
ext_style = {"f00": "C0", "nat": "C3", "f2": "C1"}
fig6, axes6 = plt.subplots(len(fields), len(cases), figsize=(14, 10), sharex=True)
for row, f in enumerate(fields):
    for col, db in enumerate(cases):
        ax = axes6[row, col]
        for k in ext_label:
            s_ = ext_df[(ext_df.db == db) & (ext_df.field == f) & (ext_df.arm == k)]
            ax.plot(s_.width, s_.log_ratio, color=ext_style[k], marker="o", ms=3, label=ext_label[k])
        ax.axhline(0, color="0.5", lw=1, marker="")
        style_x(ax)
        ax.set_title(f"{db}, {field_label[f]}")
    axes6[row, 0].set_ylabel("log(Extent Ratio)")
for ax in axes6[-1]:
    ax.set_xlabel("WIDTH")
axes6[0, 0].legend()
plt.show()

## Extent filter on top of TGLF's frequency filter (Fusion_PhD-38e9.8)

Does dropping modes whose extent is below a cutoff add accuracy to `FILTER 2` (and, for comparison, to `FILTER 0`)? On the arm's own modes, a mode is kept if it is unstable (growth > 0 after the native filter)
and its 95% extent of phi (or A_par) is >= the cutoff: absolute [rad], or a fraction of the case's largest unstable-mode extent in that arm. The dominant mode is the argmax growth among survivors; no survivor is a miss (growth 0, frequency 0, case stays in).
**One cutoff per (database, base filter, field, kind), shared by all widths, chosen in-sample on the same database, and allowed to add at most 5% of case-widths as misses over the base filter (without that limit the optimum is a cutoff that removes nearly every mode, so every case is a miss and the error is just the GS2 value)** to minimise the mean over widths of log10 medAE(gamma) + log10 medAE(omega). The cutoff is a free parameter fitted on the data it is scored on, so the gain is an upper bound. "Best" field and kind is the one with the lowest objective.

In [ ]:
ext_arms = {"f00": "FILTER 0", "f2": "FILTER 2"}
grid = [(kind, c) for kind, g_ in (("abs", cutoffs), ("rel", rel_cutoffs)) for c in g_]
pre = {}
for db in cases:
    ref = gs2[db].data
    gby, fby = dict(zip(ref.sample_name.values, mag(ref.growth_rate))), dict(zip(ref.sample_name.values, mag(ref.mode_frequency)))
    for w in widths:
        all_names = arm["f00", db, w].data.sample_name.values
        g_all = np.array([gby.get(n, np.nan) for n in all_names])
        ok = np.isfinite(g_all) & (g_all > 0)
        names = all_names[ok]
        for k in ext_arms:
            pre[k, db, w] = dict(names=names, g=grab(arm[k, db, w], "growth_rate", names), f=grab(arm[k, db, w], "mode_frequency", names),
                                 e={fl: ext_of(arm[k, db, w], fl, names) for fl in fields}, g_ref=g_all[ok], f_ref=np.array([fby[n] for n in names]))

xrows = []
for (k, db, w), p_ in pre.items():
    g, f, g_ref, f_ref = p_["g"], p_["f"], p_["g_ref"], p_["f_ref"]
    for fl in fields:
        e = p_["e"][fl]
        longest = np.nanmax(np.where(g > 0, e, np.nan), axis=1, initial=0)
        for kind, c in grid:
            thr = c if kind == "abs" else c * longest[:, None]
            gs, fs = pick(g, f, (e >= thr) | (c == 0))
            xrows.append(dict(base=k, db=db, width=width_value[w], field=fl, kind=kind, cutoff=c,
                              medAE_g=medae(gs, g_ref), medAE_f=medae(fs, f_ref), miss=int((gs == 0).sum())))
xdf = pd.DataFrame(xrows)
xdf["score"] = np.log10(xdf.medAE_g) + np.log10(xdf.medAE_f)
max_extra_miss = 0.05   # a cutoff may add at most this fraction of case-widths as misses over its base filter (else "predict nothing" wins)
obj = xdf.groupby(["base", "db", "field", "kind", "cutoff"]).agg(score=("score", "mean"), miss=("miss", "sum"), n=("miss", "size")).reset_index()
obj["n_cases"] = obj.db.map({db: int(sum(len(pre[k, db, w]["g_ref"]) for w in widths)) for db in cases for k in ["f00"]})
obj["extra"] = obj.miss - obj.groupby(["base", "db", "field", "kind"]).miss.transform("min")
obj = obj[obj.extra <= max_extra_miss * obj.n_cases]
chosen = obj.loc[obj.groupby(["base", "db", "field", "kind"]).score.idxmin()].set_index(["base", "db", "field", "kind"])
best_fk = chosen.reset_index().loc[lambda t: t.groupby(["base", "db"]).score.idxmin()].set_index(["base", "db"])[["field", "kind", "cutoff"]]
print(chosen.round(3).to_string()); print(best_fk.to_string())

In [ ]:
# at each chosen cutoff: per-width arms for the figure and the verdict table
xs = xdf.merge(chosen.cutoff.rename("c_sel").reset_index(), on=["base", "db", "field", "kind"]).query("cutoff == c_sel")
xs = xs.merge(best_fk.reset_index().rename(columns={"field": "bf", "kind": "bk", "cutoff": "bc"}), on=["base", "db"])
xbest = xs[(xs.field == xs.bf) & (xs.kind == xs.bk)]
vt = []
for db in cases:
    for base in ext_arms:
        b0 = arms_df[arms_df.db == db].set_index("width")
        for fl in fields:
            for kind in ("abs", "rel"):
                x = xs[(xs.db == db) & (xs.base == base) & (xs.field == fl) & (xs.kind == kind)].set_index("width").sort_index()
                for m in ("g", "f"):
                    a, b_ = b0[f"medAE_{m}_{base}"], x[f"medAE_{m}"]
                    vt.append(dict(db=db, base=base, field=fl, kind=kind, cutoff=chosen.loc[(base, db, fl, kind), "cutoff"], metric=m,
                                   widths_better=int((b_ < a).sum()), widths_worse=int((b_ > a).sum()), median_dlog10=float(np.median(np.log10(b_ / a))),
                                   misses_base=int(b0[f"miss_{base}"].sum()), misses_ext=int(x.miss.sum())))
vt = pd.DataFrame(vt)
print(vt.round(3).to_string(index=False))

In [ ]:
fig7, axes7 = plt.subplots(2, len(cases), figsize=(14, 10), sharex=True)
for col, db in enumerate(cases):
    s = arms_df[arms_df.db == db]
    xb = xbest[(xbest.db == db) & (xbest.base == "f2")].set_index("width").sort_index()
    for row, m in enumerate(("f", "g")):
        ax = axes7[row, col]
        ax.plot(s.width, np.log10(s[f"medAE_{m}_f00"]), color="C0", marker="o", ms=3, label="FILTER 0")
        ax.plot(s.width, np.log10(s[f"medAE_{m}_f2"]), color="C1", marker="o", ms=3, label="FILTER 2")
        ax.plot(xb.index, np.log10(xb[f"medAE_{m}"]), color="C2", marker="o", ms=3, label="FILTER 2 + extent")
        ax.plot(s.width, np.log10(s[f"medAE_{m}_orc"]), color="0.5", ls=":", label="oracle")
        style_x(ax)
        ax.set_ylabel(ylab[m])
    axes7[0, col].set_title(db)
    axes7[1, col].set_xlabel("WIDTH")
axes7[1, 0].legend()
plt.show()

In [ ]:
# extent of the selected mode relative to GS2 (as fig6): FILTER 2 vs FILTER 2 + the extent filter of fig7
xr = []
for db in cases:
    bf, bk, bc = best_fk.loc[("f2", db), ["field", "kind", "cutoff"]]
    for w in widths:
        p_ = pre["f2", db, w]
        g, r = p_["g"], np.arange(len(p_["g"]))
        longest = np.nanmax(np.where(g > 0, p_["e"][bf], np.nan), axis=1, initial=0)
        thr = bc if bk == "abs" else bc * longest[:, None]
        for lab, keep in (("FILTER 2", True), ("FILTER 2 + extent", p_["e"][bf] >= thr)):
            gsel, i = select(g, keep)
            for fl in fields:
                e_ref = ext_of(gs2[db], fl, p_["names"])[:, 0]
                with np.errstate(divide="ignore", invalid="ignore"):
                    lr = np.log10(p_["e"][fl][r, i] / e_ref)
                xr.append(dict(db=db, width=width_value[w], field=fl, arm=lab, log_ratio=float(np.nanmedian(np.where(gsel > 0, lr, np.nan)))))
xr = pd.DataFrame(xr)
fig8, axes8 = plt.subplots(len(fields), len(cases), figsize=(14, 10), sharex=True)
for row, fl in enumerate(fields):
    for col, db in enumerate(cases):
        ax = axes8[row, col]
        for lab, color in (("FILTER 2", "C1"), ("FILTER 2 + extent", "C2")):
            s_ = xr[(xr.db == db) & (xr.field == fl) & (xr.arm == lab)]
            ax.plot(s_.width, s_.log_ratio, color=color, marker="o", ms=3, label=lab)
        ax.axhline(0, color="0.5", lw=1)
        style_x(ax)
        ax.set_title(f"{db}, {field_label[fl]}")
    axes8[row, 0].set_ylabel("log(Extent Ratio)")
for ax in axes8[-1]:
    ax.set_xlabel("WIDTH")
axes8[0, 0].legend()
plt.show()
print(xr.pivot_table(index=["db", "field", "arm"], values="log_ratio", aggfunc=["median", "min", "max"]).round(3))
print(xr.pivot(index=["db", "field", "width"], columns="arm", values="log_ratio").assign(d=lambda t: t["FILTER 2 + extent"] - t["FILTER 2"]).groupby(["db", "field"]).d.agg(["median", "min", "max"]).round(3))


## Save

In [ ]:
output_dir = ROOT / "Plots" / analysis_name
output_dir.mkdir(parents=True, exist_ok=True)
fig6.savefig(output_dir / "fig6_extent_ratio_vs_width.png", bbox_inches="tight")
fig5.savefig(output_dir / "fig5_filters_medae_vs_width.png", bbox_inches="tight")
summ.round(4).to_csv(output_dir / "filter_summary.csv", index=False)
removed_df.to_csv(output_dir / "modes_removed.csv", index=False)
arms_df.round(5).to_csv(output_dir / "medae_by_width.csv", index=False)
fig7.savefig(output_dir / "fig7_extent_filter_on_filter2.png", bbox_inches="tight")
vt.round(4).to_csv(output_dir / "extent_filter_summary.csv", index=False)
chosen.round(4).to_csv(output_dir / "extent_filter_cutoffs.csv")
fig8.savefig(output_dir / "fig8_extent_ratio_with_extent_filter.png", bbox_inches="tight")
